In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

!pip install pytorch-metric-learning -q
print('Done.')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 5.6 MB/s eta 0:00:00
Done.


In [ ]:
import os

# ── Paste your Kaggle credentials here ───────────────────────────────────────
KAGGLE_USERNAME = 'saptarshisarkar30'
KAGGLE_KEY      = 'KGAT_837c6eb475da0a8f51f276c7573f547e'
# ─────────────────────────────────────────────────────────────────────────────

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')

!pip install kaggle -q
!kaggle datasets download -d majdouline20/shapenetpart-dataset
!unzip -q shapenetpart-dataset.zip

os.chdir('/content')

# Auto-find the PartAnnotation folder
DATA_ROOT = None
for root, dirs, files in os.walk('/content/data'):
    if '03001627' in dirs:  # chair synset
        DATA_ROOT = root
        break

if DATA_ROOT:
    print(f'Dataset found at: {DATA_ROOT}')
else:
    print('ERROR: Could not find dataset. Check unzip output above.')

Dataset URL: https://www.kaggle.com/datasets/majdouline20/shapenetpart-dataset
License(s): MIT
100% 1.02G/1.02G [00:10<00:00, 106MB/s]

Dataset found at: /content/data/PartAnnotation


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import jaccard_score, fbeta_score, precision_score, recall_score
from pytorch_metric_learning.losses import NTXentLoss
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
CFG = {
    # Data
    'data_root'      : DATA_ROOT,   # set automatically in Cell 2
    'obj_class'      : 4,           # 4 = chair
    'num_points'     : 1024,
    'train_ratio'    : 0.8,

    # Training
    'batch_size'     : 8,
    'n_epochs'       : 50,          # 50 is enough — model peaks early
    'lr'             : 3e-4,        # stable learning rate
    'step_size'      : 15,          # StepLR: decay every 15 epochs
    'gamma'          : 0.5,         # StepLR: multiply LR by 0.5

    # Model
    'n_fg'           : 256,         # foreground sample points
    'n_bg'           : 256,         # background sample points
    'group_size'     : 7,
    'emb_dim'        : 512,         # DGCNN embedding dimension
    'dgcnn_k'        : 20,          # DGCNN KNN graph

    # Loss weights
    'repulsion'      : 0.5,
    'alpha'          : 1.0,         # samplenet loss weight
    'n_pos_pairs'    : 300,
    'n_neg_pairs'    : 600,
    'temp_point'     : 0.5,
    'temp_obj'       : 0.07,

    # Professor's spatial loss
    'lambda_spatial' : 0.1,         # try 0.01, 0.1, 0.5
    'k_spatial'      : 10,          # KNN neighbours in XYZ space
}

CLASS_TO_NAME = {
    0:'airplane', 1:'bag',      2:'cap',        3:'car',
    4:'chair',    5:'earphone',  6:'guitar',     7:'knife',
    8:'lamp',     9:'laptop',   10:'motorbike', 11:'mug',
   12:'pistol',  13:'rocket',  14:'skateboard', 15:'table'
}

print('Config set.')
print(f'Training on: {CLASS_TO_NAME[CFG["obj_class"]]}')
print(f'Epochs: {CFG["n_epochs"]} | LR: {CFG["lr"]} | Batch: {CFG["batch_size"]}')

Config set.
Training on: chair
Epochs: 50 | LR: 0.0003 | Batch: 8


In [ ]:
SYNSET_TO_CLASS = {
    '02691156': 0,  '02773838': 1,  '02954340': 2,  '02958343': 3,
    '03001627': 4,  '03261776': 5,  '03467517': 6,  '03624134': 7,
    '03636649': 8,  '03642806': 9,  '03790512': 10, '03797390': 11,
    '03948459': 12, '04099429': 13, '04225987': 14, '04379243': 15,
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3:
                pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    with open(path) as f:
        return np.array([int(l.strip()) for l in f if l.strip()], dtype='int64')

class ShapeNetPart(Dataset):
    def __init__(self, data_root, obj_class=4, partition='train',
                 num_points=1024, train_ratio=0.8):
        self.num_points = num_points

        synset  = next(s for s, c in SYNSET_TO_CLASS.items() if c == obj_class)
        syn_dir = os.path.join(data_root, synset)
        pts_dir = os.path.join(syn_dir, 'points')
        seg_dir = os.path.join(syn_dir, 'points_label')

        # Build seg map (handles flat or nested layout)
        seg_map = {}
        if os.path.isdir(seg_dir):
            for entry in os.listdir(seg_dir):
                ep = os.path.join(seg_dir, entry)
                if entry.endswith('.seg'):
                    seg_map[entry[:-4]] = ep
                elif os.path.isdir(ep):
                    for f in os.listdir(ep):
                        if f.endswith('.seg') and f[:-4] not in seg_map:
                            seg_map[f[:-4]] = os.path.join(ep, f)

        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        valid   = [i for i in all_ids if i in seg_map]

        if len(valid) == 0:
            raise RuntimeError(f'No matched pts+seg pairs found in {syn_dir}')

        np.random.seed(42)
        perm  = np.random.permutation(len(valid))
        split = int(len(valid) * train_ratio)
        chosen = [valid[i] for i in (perm[:split] if partition == 'train' else perm[split:])]

        self.samples = [(os.path.join(pts_dir, s+'.pts'), seg_map[s]) for s in chosen]
        print(f'[ShapeNetPart] {CLASS_TO_NAME[obj_class]} | {partition} | {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc), len(seg))
        pc, seg = pc[:n], seg[:n]

        if n >= self.num_points:
            idx_s = np.random.choice(n, self.num_points, replace=False)
        else:
            idx_s = np.random.choice(n, self.num_points, replace=True)
        pc, seg = pc[idx_s], seg[idx_s]

        # Normalize to unit sphere
        pc -= pc.mean(0)
        s   = np.max(np.linalg.norm(pc, axis=1))
        if s > 0: pc /= s

        # Binary label: 0=background, 1=foreground
        binary = (seg > seg.min()).astype('int64')
        return pc.astype('float32'), binary


# Build datasets
train_ds = ShapeNetPart(CFG['data_root'], CFG['obj_class'], 'train',
                        CFG['num_points'], CFG['train_ratio'])
test_ds  = ShapeNetPart(CFG['data_root'], CFG['obj_class'], 'test',
                        CFG['num_points'], CFG['train_ratio'])

bs = min(CFG['batch_size'], len(train_ds))
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                          drop_last=True, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=1,  shuffle=False,
                          drop_last=False, num_workers=2)

print(f'Train batches : {len(train_loader)}')
print(f'Test  samples : {len(test_ds)}')
pc, lbl = train_ds[0]
print(f'Sample shape  : {pc.shape} | Labels: {set(lbl.tolist())}')

[ShapeNetPart] chair | train | 5422 samples
[ShapeNetPart] chair | test | 1356 samples
Train batches : 677
Test  samples : 1356
Sample shape  : (1024, 3) | Labels: {0, 1}


In [ ]:
def knn_graph(x, k):
    """x: (B,D,N) -> idx: (B,N,k)  — pure PyTorch, no CUDA extensions"""
    xt   = x.permute(0, 2, 1)                                    # (B,N,D)
    dist = torch.cdist(xt, xt)                                    # (B,N,N)
    return dist.topk(k+1, dim=-1, largest=False).indices[:,:,1:] # (B,N,k)

def get_graph_feature(x, k=20):
    """Edge-conditioned graph feature for DGCNN."""
    B, D, N = x.shape
    idx      = knn_graph(x, k)                                    # (B,N,k)
    idx_base = torch.arange(B, device=x.device).view(-1,1,1) * N
    idx_flat = (idx + idx_base).view(-1)                          # (B*N*k,)
    xt  = x.permute(0,2,1).contiguous()                           # (B,N,D)
    nbr = xt.view(B*N, -1)[idx_flat].view(B, N, k, D)             # (B,N,k,D)
    ctr = xt.view(B, N, 1, D).expand(B, N, k, D)                  # (B,N,k,D)
    feat = torch.cat([nbr-ctr, ctr], dim=3)                       # (B,N,k,2D)
    return feat.permute(0,3,1,2).contiguous()                     # (B,2D,N,k)

class DGCNN(nn.Module):
    """
    Dynamic Graph CNN feature extractor.
    Input:  (B, 3, N)
    Output: point_feat (B, emb_dim, N),  obj_feat (B, 256)
    """
    def __init__(self, k=20, emb_dim=512):
        super().__init__()
        self.k = k

        self.conv1 = nn.Sequential(
            nn.Conv2d(6,   64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(
            nn.Conv1d(512, emb_dim, 1, bias=False),
            nn.BatchNorm1d(emb_dim), nn.LeakyReLU(0.2))

        self.head = nn.Sequential(
            nn.Linear(emb_dim*2, 512, bias=False),
            nn.BatchNorm1d(512), nn.LeakyReLU(0.2), nn.Dropout(0.4),
            nn.Linear(512, 256, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.4),
        )
        self.emb_dim = emb_dim

    def forward(self, x):
        """x: (B,3,N)"""
        x1 = self.conv1(get_graph_feature(x,  self.k)).max(-1)[0]  # (B,64,N)
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(-1)[0]  # (B,64,N)
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(-1)[0]  # (B,128,N)
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(-1)[0]  # (B,256,N)

        cat        = torch.cat([x1,x2,x3,x4], dim=1)  # (B,512,N)
        point_feat = self.conv5(cat)                    # (B,emb_dim,N)

        gmp = F.adaptive_max_pool1d(point_feat, 1).squeeze(-1)  # (B,emb_dim)
        gap = F.adaptive_avg_pool1d(point_feat, 1).squeeze(-1)  # (B,emb_dim)
        obj_feat = self.head(torch.cat([gmp, gap], 1))           # (B,256)

        return point_feat, obj_feat

print('DGCNN defined.')

DGCNN defined.


In [ ]:
class PointSampler(nn.Module):
    """
    Learns to sample num_out points from input cloud.
    Uses soft projection to snap output onto real input points.
    """
    def __init__(self, num_out=256, bottleneck=128, group_size=7):
        super().__init__()
        self.num_out    = num_out
        self.group_size = group_size

        self.encoder = nn.Sequential(
            nn.Conv1d(3,   64,  1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, bottleneck, 1), nn.BatchNorm1d(bottleneck), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 512),        nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, num_out * 3),
        )
        self.sigma = nn.Parameter(torch.tensor(1.0))
        self._proj_loss = None

    def forward(self, x):
        """
        x: (B, N, 3)
        returns simp_pc (B,M,3), proj_pc (B,M,3)
        """
        B, N, _ = x.shape

        # Encode
        h    = self.encoder(x.permute(0,2,1)).max(-1)[0]  # (B, bottleneck)
        simp = self.decoder(h).view(B, self.num_out, 3)   # (B, M, 3)

        # Soft projection: snap each simplified point onto nearest real points
        dist = torch.cdist(simp, x)                        # (B, M, N)
        g    = min(self.group_size, N)
        topk = dist.topk(g, dim=-1, largest=False)
        kd, ki = topk.values, topk.indices                 # (B, M, g)

        ki_e = ki.unsqueeze(-1).expand(B, self.num_out, g, 3)
        nbr  = x.unsqueeze(1).expand(B, self.num_out, N, 3)
        nbr  = torch.gather(nbr, 2, ki_e)                  # (B, M, g, 3)

        w    = torch.softmax(-kd / self.sigma.clamp(min=1e-2), dim=-1)  # (B,M,g)
        proj = (w.unsqueeze(-1) * nbr).sum(2)              # (B, M, 3)

        self._proj_loss = ((simp - proj)**2).sum(-1).mean()
        return simp, proj

    def get_simplification_loss(self, pc, simp, n, gamma=1.0):
        return torch.cdist(simp, pc).min(-1)[0].mean()

    def get_projection_loss(self):
        return self._proj_loss if self._proj_loss is not None else torch.tensor(0.0)


class CoSegNet(nn.Module):
    """Two samplers: one for foreground, one for background."""
    def __init__(self, n_fg=256, n_bg=256, group_size=7):
        super().__init__()
        self.fg = PointSampler(n_fg, group_size=group_size)
        self.bg = PointSampler(n_bg, group_size=group_size)

print('CoSegNet defined.')

CoSegNet defined.


In [ ]:
def chamfer_distance(pc1, pc2):
    """
    Pure PyTorch Chamfer Distance.
    pc1, pc2: (B, N, 3)
    returns d1 (B,N), d2 (B,M) — nearest neighbour distances
    """
    d = torch.cdist(pc1, pc2)   # (B, N, M)
    return d.min(2)[0], d.min(1)[0]


def spatial_consistency_loss(point_features, xyz, k=10):
    """
    ── PROFESSOR'S EXTENSION ──────────────────────────────────────────────────
    Encourages spatially neighboring points to have similar feature embeddings.
    This produces smoother segmentation boundaries after clustering.

    L_spatial = mean over all (i, j) where j ∈ KNN_k(i) of ||F_i - F_j||^2

    KNN is computed in XYZ space (Euclidean distance).
    Features F are the per-point embeddings from DGCNN.

    Args:
        point_features : (B, D, N)  per-point embeddings from DGCNN
        xyz            : (B, N, 3)  XYZ coordinates of the same points
        k              : int        number of spatial nearest neighbours

    Returns:
        scalar loss value
    ──────────────────────────────────────────────────────────────────────────
    """
    B, D, N = point_features.shape

    # Step 1: find k nearest neighbours in XYZ space
    dist    = torch.cdist(xyz, xyz)                                   # (B,N,N)
    knn_idx = dist.topk(k+1, dim=-1, largest=False).indices[:,:,1:]  # (B,N,k)

    # Step 2: gather neighbour features
    feats    = point_features.permute(0,2,1).contiguous()  # (B,N,D)
    flat     = knn_idx.reshape(B, -1)                       # (B,N*k)
    flat_exp = flat.unsqueeze(-1).expand(B, N*k, D)         # (B,N*k,D)
    nbr_f    = torch.gather(feats, 1, flat_exp).view(B,N,k,D)  # (B,N,k,D)

    # Step 3: compute squared feature differences
    fi   = feats.unsqueeze(2).expand(B, N, k, D)           # (B,N,k,D)
    loss = ((fi - nbr_f)**2).sum(-1).mean()
    return loss


print('Loss functions defined.')
print('  - chamfer_distance')
print('  - spatial_consistency_loss  <-- professor extension')

Loss functions defined.
  - chamfer_distance
  - spatial_consistency_loss  <-- professor extension


In [ ]:
def fps(pc, n):
    """Farthest point sampling. pc: (B,N,3) -> (B,n,3)"""
    B, N, _ = pc.shape
    if n >= N: return pc
    dev   = pc.device
    cents = torch.zeros(B, n, dtype=torch.long, device=dev)
    dist  = torch.full((B,N), 1e10, device=dev)
    far   = torch.randint(0, N, (B,), device=dev)
    for i in range(n):
        cents[:,i] = far
        c    = pc[torch.arange(B, device=dev), far].unsqueeze(1)
        d    = ((pc - c)**2).sum(-1)
        dist = torch.min(dist, d)
        far  = dist.argmax(-1)
    idx = cents.unsqueeze(-1).expand(B, n, 3)
    return torch.gather(pc, 1, idx)


def run_training(use_spatial=False, tag='baseline'):
    """
    use_spatial = False  ->  L = L_contrastive              (baseline)
    use_spatial = True   ->  L = L_contrastive + λ*L_spatial (extension)

    Returns history dict with iou, f1, loss per epoch.
    """
    sep = '='*60
    print(f'\n{sep}')
    print(f'  {tag.upper()}  |  spatial_loss={use_spatial}')
    if use_spatial:
        print(f'  lambda={CFG["lambda_spatial"]}  k={CFG["k_spatial"]}')
    print(f'{sep}\n')

    save_dir = f'/content/results/{tag}'
    os.makedirs(save_dir, exist_ok=True)
    log_path = f'{save_dir}/log.txt'

    def log(msg):
        print(msg)
        with open(log_path, 'a') as f:
            f.write(msg + '\n')

    # ── Models ────────────────────────────────────────────────────────────────
    net  = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['group_size']).to(DEVICE)
    feat = nn.DataParallel(DGCNN(k=CFG['dgcnn_k'], emb_dim=CFG['emb_dim'])).to(DEVICE)

    # Optimise sampler only — feature extractor trains jointly
    all_params = list(net.parameters()) + list(feat.parameters())
    opt   = optim.AdamW(all_params, lr=CFG['lr'], weight_decay=1e-4)
    sched = StepLR(opt, step_size=CFG['step_size'], gamma=CFG['gamma'])

    pt_loss_fn  = NTXentLoss(temperature=CFG['temp_point'])
    obj_loss_fn = NTXentLoss(temperature=CFG['temp_obj'])

    history = {'iou':[], 'f1':[], 'loss':[], 'precision':[], 'recall':[]}
    best_f1  = 0.0
    best_iou = 0.0

    for epoch in range(CFG['n_epochs']):
        net.train()
        feat.train()
        running = 0.0
        nb      = 0

        for coord, label in train_loader:
            coord = coord.to(DEVICE)                        # (B,N,3)
            coord = fps(coord, CFG['num_points'])

            # Sample FG and BG points
            fg_simp, fg_coord = net.fg(coord)              # (B, n_fg, 3)
            bg_simp, bg_coord = net.bg(coord)              # (B, n_bg, 3)

            # Repulsion loss — keep FG and BG apart
            d1, d2   = chamfer_distance(fg_coord, bg_coord)
            rep_loss = (torch.clamp(CFG['repulsion'] - d1, min=0).mean() +
                        torch.clamp(CFG['repulsion'] - d2, min=0).mean())

            # Extract features
            fg_pt_feat, fg_obj = feat(fg_coord.permute(0,2,1).contiguous())
            bg_pt_feat, bg_obj = feat(bg_coord.permute(0,2,1).contiguous())

            # SampleNet loss
            samp_loss = CFG['alpha'] * (
                net.fg.get_simplification_loss(coord, fg_simp, CFG['n_fg']) +
                net.fg.get_projection_loss() +
                net.bg.get_simplification_loss(coord, bg_simp, CFG['n_bg']) +
                net.bg.get_projection_loss()
            )

            # Point-level contrastive loss
            B = coord.shape[0]
            pt_losses = []
            for i in range(B):
                of  = fg_pt_feat[i].permute(1,0)  # (n_fg, D)
                bf  = bg_pt_feat[i].permute(1,0)  # (n_bg, D)
                emb = torch.cat([of, bf], 0)
                lbl = torch.cat([
                    torch.zeros(of.shape[0]),
                    torch.arange(1, bf.shape[0]+1)
                ]).long().to(DEVICE)
                pos = torch.randint(0, of.shape[0], (CFG['n_pos_pairs'],2), device=DEVICE)
                neg = torch.randint(0, bf.shape[0], (CFG['n_neg_pairs'],2), device=DEVICE)
                neg[:,1] += of.shape[0]
                tup = (pos[:,0], pos[:,1], neg[:,0], neg[:,1])
                pt_losses.append(pt_loss_fn(emb, lbl, tup))
            pt_loss = torch.stack(pt_losses).mean()

            # Object-level contrastive loss
            emb = torch.cat([fg_obj, bg_obj], 0)
            lbl = torch.cat([
                torch.zeros(fg_obj.shape[0]),
                torch.arange(1, bg_obj.shape[0]+1)
            ]).long().to(DEVICE)
            obj_loss = obj_loss_fn(emb, lbl)

            # ── TOTAL LOSS ────────────────────────────────────────────────────
            if use_spatial:
                # Professor's extension
                sp_loss = spatial_consistency_loss(
                    fg_pt_feat, fg_coord, k=CFG['k_spatial']
                )
                loss = (pt_loss + obj_loss + samp_loss + rep_loss
                        + CFG['lambda_spatial'] * sp_loss)
            else:
                # Baseline
                loss = pt_loss + obj_loss + samp_loss + rep_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)  # gradient clip
            opt.step()
            running += loss.item()
            nb      += 1

        sched.step()

        # ── VALIDATION ────────────────────────────────────────────────────────
        net.eval()
        feat.eval()
        y_preds, y_trues = [], []

        with torch.no_grad():
            for coord, label in test_loader:
                coord = coord.to(DEVICE)             # (1,N,3)
                coord = fps(coord, CFG['num_points'])

                _, fg_coord = net.fg(coord)           # (1,n_fg,3)

                # Each input point: assign FG if close to any FG sample
                dist   = torch.cdist(coord, fg_coord).squeeze(0)  # (N, n_fg)
                min_d  = dist.min(-1)[0]                           # (N,)
                thresh = min_d.topk(CFG['n_fg'], largest=False).values[-1]
                pred   = (min_d <= thresh).long().cpu().numpy()    # (N,)

                true = label.squeeze().numpy()
                y_preds.append(pred)
                y_trues.append(true)

        yp = np.concatenate(y_preds)
        yt = np.concatenate(y_trues)

        iou = jaccard_score(yt, yp,        zero_division=0)
        f1  = fbeta_score(yt,  yp, beta=0.75, zero_division=0)
        pre = precision_score(yt, yp,      zero_division=0)
        rec = recall_score(yt,    yp,      zero_division=0)
        avg_loss = running / nb

        history['iou'].append(iou)
        history['f1'].append(f1)
        history['loss'].append(avg_loss)
        history['precision'].append(pre)
        history['recall'].append(rec)

        msg = (f'[{tag}] Epoch {epoch:03d} | '
               f'loss {avg_loss:.4f} | '
               f'IOU {iou:.4f} | F1 {f1:.4f} | '
               f'P {pre:.3f} | R {rec:.3f}')
        log(msg)

        if f1 > best_f1:
            best_f1  = f1
            best_iou = iou
            torch.save(net.state_dict(),  f'{save_dir}/model_best.pt')
            torch.save(feat.state_dict(), f'{save_dir}/feat_best.pt')
            log(f'  --> Best saved: IOU={best_iou:.4f}  F1={best_f1:.4f}')

    summary = (f'\nFINISHED [{tag}] | '
               f'Best IOU={best_iou:.4f} | Best F1={best_f1:.4f}')
    log(summary)
    return history


print('Training function ready.')

Training function ready.


In [ ]:
history_baseline = run_training(use_spatial=False, tag='baseline')


  BASELINE  |  spatial_loss=False

[baseline] Epoch 000 | loss 5.9742 | IOU 0.2422 | F1 0.4107 | P 0.476 | R 0.330
  --> Best saved: IOU=0.2422  F1=0.4107
[baseline] Epoch 001 | loss 5.5927 | IOU 0.2165 | F1 0.3749 | P 0.434 | R 0.302
[baseline] Epoch 002 | loss 4.0176 | IOU 0.2499 | F1 0.4212 | P 0.488 | R 0.339
  --> Best saved: IOU=0.2499  F1=0.4212
[baseline] Epoch 003 | loss 2.9738 | IOU 0.3063 | F1 0.4939 | P 0.572 | R 0.397
  --> Best saved: IOU=0.3063  F1=0.4939
[baseline] Epoch 004 | loss 2.5895 | IOU 0.3627 | F1 0.5606 | P 0.649 | R 0.451
  --> Best saved: IOU=0.3627  F1=0.5606
[baseline] Epoch 005 | loss 2.3955 | IOU 0.3910 | F1 0.5920 | P 0.686 | R 0.476
  --> Best saved: IOU=0.3910  F1=0.5920
[baseline] Epoch 006 | loss 2.2823 | IOU 0.4228 | F1 0.6259 | P 0.725 | R 0.504
  --> Best saved: IOU=0.4228  F1=0.6259
[baseline] Epoch 007 | loss 2.1419 | IOU 0.4166 | F1 0.6193 | P 0.717 | R 0.499
[baseline] Epoch 008 | loss 2.0272 | IOU 0.3998 | F1 0.6017 | P 0.697 | R 0.484
[bas

In [ ]:
history_spatial = run_training(use_spatial=True, tag='spatial_loss')


  SPATIAL_LOSS  |  spatial_loss=True
  lambda=0.1  k=10

[spatial_loss] Epoch 000 | loss 6.4049 | IOU 0.2530 | F1 0.4252 | P 0.492 | R 0.342
  --> Best saved: IOU=0.2530  F1=0.4252
[spatial_loss] Epoch 001 | loss 4.0418 | IOU 0.3738 | F1 0.5730 | P 0.663 | R 0.461
  --> Best saved: IOU=0.3738  F1=0.5730
[spatial_loss] Epoch 002 | loss 3.1919 | IOU 0.3436 | F1 0.5386 | P 0.624 | R 0.433
[spatial_loss] Epoch 003 | loss 2.8671 | IOU 0.3466 | F1 0.5421 | P 0.628 | R 0.436
[spatial_loss] Epoch 004 | loss 2.6705 | IOU 0.3608 | F1 0.5583 | P 0.646 | R 0.449
[spatial_loss] Epoch 005 | loss 2.5262 | IOU 0.3734 | F1 0.5726 | P 0.663 | R 0.461
[spatial_loss] Epoch 006 | loss 2.4252 | IOU 0.3802 | F1 0.5801 | P 0.671 | R 0.467
  --> Best saved: IOU=0.3802  F1=0.5801
[spatial_loss] Epoch 007 | loss 2.3592 | IOU 0.3760 | F1 0.5754 | P 0.666 | R 0.463
[spatial_loss] Epoch 008 | loss 2.2791 | IOU 0.3540 | F1 0.5505 | P 0.637 | R 0.443
[spatial_loss] Epoch 009 | loss 2.2128 | IOU 0.3589 | F1 0.5563 | 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'Baseline vs +Spatial Loss | ShapeNet Part — {CLASS_TO_NAME[CFG["obj_class"]]}',
    fontsize=14, fontweight='bold'
)

epochs = range(1, CFG['n_epochs']+1)
labels = ['Baseline', '+Spatial Loss (λ={})'.format(CFG['lambda_spatial'])]
colors = ['steelblue', 'darkorange']
hists  = [history_baseline, history_spatial]

for ax, key, title in zip(axes,
                           ['iou',   'f1',       'loss'],
                           ['IOU',   'F1 Score', 'Training Loss']):
    for h, lbl, col in zip(hists, labels, colors):
        ax.plot(epochs, h[key], label=lbl, color=col, linewidth=2)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('/content/results', exist_ok=True)
plt.savefig('/content/results/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /content/results/comparison.png')

# ── Summary table ─────────────────────────────────────────────────────────────
b_iou = max(history_baseline['iou'])
b_f1  = max(history_baseline['f1'])
s_iou = max(history_spatial['iou'])
s_f1  = max(history_spatial['f1'])

print('\n' + '='*58)
print(f'{"Metric":<20} {"Baseline":>17} {"+ Spatial Loss":>17}')
print('='*58)
print(f'{"Best IOU":<20} {b_iou:>17.4f} {s_iou:>17.4f}')
print(f'{"Best F1":<20} {b_f1:>17.4f} {s_f1:>17.4f}')
print(f'{"Final Loss":<20} {history_baseline["loss"][-1]:>17.4f} {history_spatial["loss"][-1]:>17.4f}')
print('='*58)
print(f'IOU improvement : {s_iou - b_iou:+.4f}')
print(f'F1  improvement : {s_f1  - b_f1:+.4f}')

In [ ]:
ablation_results = {}

for lam in [0.01, 0.5]:
    CFG['lambda_spatial'] = lam
    tag = f'spatial_lambda_{str(lam).replace(".","p")}'
    ablation_results[lam] = run_training(use_spatial=True, tag=tag)

# Reset to default
CFG['lambda_spatial'] = 0.1

# Print ablation summary
print('\n' + '='*50)
print(f'{"Lambda":>10} {"Best IOU":>15} {"Best F1":>15}')
print('='*50)
print(f'{"Baseline":>10} {max(history_baseline["iou"]):>15.4f} {max(history_baseline["f1"]):>15.4f}')
print(f'{0.01:>10} {max(ablation_results[0.01]["iou"]):>15.4f} {max(ablation_results[0.01]["f1"]):>15.4f}')
print(f'{0.1:>10} {max(history_spatial["iou"]):>15.4f} {max(history_spatial["f1"]):>15.4f}')
print(f'{0.5:>10} {max(ablation_results[0.5]["iou"]):>15.4f} {max(ablation_results[0.5]["f1"]):>15.4f}')
print('='*50)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/BTP_coseg_results'
os.makedirs(dest, exist_ok=True)
shutil.copytree('/content/results', dest, dirs_exist_ok=True)

print(f'Saved to Google Drive: {dest}')
print('Files:')
for root, dirs, files in os.walk(dest):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) // 1024
        print(f'  {path.replace(dest,"")}  ({size} KB)')